# Phase 2 - Data Preparation and Feature Engineering


## Objectives

- Clean missing values and known anomalous values.
- Engineer credit-risk features from application data.
- Encode categorical variables.
- Create stratified train, validation, and test splits.
- Save reproducible model-ready outputs.

In [18]:
from pathlib import Path
import json

import pandas as pd

DATA_PATH = "D:/DataScience_project/SCB_intern_project/outputs/applications_dataset.csv"
OUTPUT_DIR = Path("outputs/phase_2")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TARGET = "TARGET"
ID_COLUMN = "SK_ID_CURR"
RANDOM_STATE = 42
MISSING_THRESHOLD = 0.65
VALIDATION_SIZE = 0.15
TEST_SIZE = 0.15

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 180)

## Load Data

In [19]:
raw = pd.read_csv(DATA_PATH)
raw.shape

(307511, 122)

In [20]:
raw[[ID_COLUMN, TARGET]].head()

,SK_ID_CURR,TARGET
0,100002,1
1,100003,0
2,100004,0
3,100006,0
4,100007,0


## Feature Engineering Helpers

In [21]:
def safe_divide(numerator: pd.Series, denominator: pd.Series) -> pd.Series:
    denominator = denominator.replace(0, pd.NA)
    return numerator / denominator


def add_features(df: pd.DataFrame) -> pd.DataFrame:
    result = df.copy()

    if "DAYS_EMPLOYED" in result.columns:
        result["DAYS_EMPLOYED_ANOMALY"] = (result["DAYS_EMPLOYED"] == 365243).astype(int)
        result.loc[result["DAYS_EMPLOYED"] == 365243, "DAYS_EMPLOYED"] = pd.NA

    if "DAYS_BIRTH" in result.columns:
        result["AGE_YEARS"] = (-result["DAYS_BIRTH"] / 365.25).clip(lower=0)

    if "DAYS_EMPLOYED" in result.columns:
        result["EMPLOYMENT_YEARS"] = (-result["DAYS_EMPLOYED"] / 365.25).clip(lower=0)

    if {"AMT_CREDIT", "AMT_INCOME_TOTAL"}.issubset(result.columns):
        result["CREDIT_INCOME_RATIO"] = safe_divide(result["AMT_CREDIT"], result["AMT_INCOME_TOTAL"])

    if {"AMT_ANNUITY", "AMT_INCOME_TOTAL"}.issubset(result.columns):
        result["ANNUITY_INCOME_RATIO"] = safe_divide(result["AMT_ANNUITY"], result["AMT_INCOME_TOTAL"])

    if {"AMT_ANNUITY", "AMT_CREDIT"}.issubset(result.columns):
        result["CREDIT_TERM_YEARS"] = safe_divide(result["AMT_CREDIT"], result["AMT_ANNUITY"]) / 12

    if {"AMT_GOODS_PRICE", "AMT_CREDIT"}.issubset(result.columns):
        result["GOODS_CREDIT_RATIO"] = safe_divide(result["AMT_GOODS_PRICE"], result["AMT_CREDIT"])

    external_sources = [column for column in ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"] if column in result]
    if external_sources:
        result["EXT_SOURCE_MEAN"] = result[external_sources].mean(axis=1)
        result["EXT_SOURCE_MIN"] = result[external_sources].min(axis=1)
        result["EXT_SOURCE_MAX"] = result[external_sources].max(axis=1)
        result["EXT_SOURCE_MISSING_COUNT"] = result[external_sources].isna().sum(axis=1)

    document_columns = [column for column in result.columns if column.startswith("FLAG_DOCUMENT_")]
    if document_columns:
        result["DOCUMENTS_PROVIDED_COUNT"] = result[document_columns].sum(axis=1)

    if "AMT_REQ_CREDIT_BUREAU_YEAR" in result.columns:
        result["CREDIT_BUREAU_REQUESTS_LAST_YEAR"] = result["AMT_REQ_CREDIT_BUREAU_YEAR"]

    return result

## Clean and Drop High-Missing Columns

In [24]:
prepared = add_features(raw)
protected_columns = {ID_COLUMN, TARGET}
missing_rates = prepared.isna().mean()
dropped_columns = [
    column
    for column, missing_rate in missing_rates.items()
    if missing_rate > MISSING_THRESHOLD and column not in protected_columns
]
prepared = prepared.drop(columns=dropped_columns)

print(f"Dropped high-missing columns: {len(dropped_columns)}")
dropped_columns[:30]

Dropped high-missing columns: 17


['OWN_CAR_AGE',
 'YEARS_BUILD_AVG',
 'COMMONAREA_AVG',
 'FLOORSMIN_AVG',
 'LIVINGAPARTMENTS_AVG',
 'NONLIVINGAPARTMENTS_AVG',
 'YEARS_BUILD_MODE',
 'COMMONAREA_MODE',
 'FLOORSMIN_MODE',
 'LIVINGAPARTMENTS_MODE',
 'NONLIVINGAPARTMENTS_MODE',
 'YEARS_BUILD_MEDI',
 'COMMONAREA_MEDI',
 'FLOORSMIN_MEDI',
 'LIVINGAPARTMENTS_MEDI',
 'NONLIVINGAPARTMENTS_MEDI',
 'FONDKAPREMONT_MODE']

### Why Dropping High-Missing Columns Is Acceptable Here

This cell drops columns where more than `MISSING_THRESHOLD` of values are missing. The default threshold is 65%, meaning a column is removed only when most applicants do not have that information.

This is acceptable for Phase 2 because:

- very high-missing columns contain limited usable information for most applicants;
- many of the dropped columns are detailed housing/building attributes, which are not consistently available;
- keeping many sparse columns can add noise and make the model harder to train;
- imputing columns with mostly missing values can create artificial patterns that are not reliable;
- protected columns like `SK_ID_CURR` and `TARGET` are never dropped by this rule.

This is a conservative first modeling dataset. Later, if a high-missing column appears business-critical, we can bring it back with a missing-value indicator and test whether it improves validation performance.

## Stratified Train, Validation, Test Split

In [25]:
try:
    from sklearn.model_selection import train_test_split
except ImportError:
    train_test_split = None

if train_test_split is None:
    shuffled = prepared.sample(frac=1, random_state=RANDOM_STATE)
    test_count = int(len(shuffled) * TEST_SIZE)
    validation_count = int(len(shuffled) * VALIDATION_SIZE)
    test = shuffled.iloc[:test_count]
    validation = shuffled.iloc[test_count : test_count + validation_count]
    train = shuffled.iloc[test_count + validation_count :]
else:
    train_validation, test = train_test_split(
        prepared,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=prepared[TARGET],
    )
    adjusted_validation_size = VALIDATION_SIZE / (1 - TEST_SIZE)
    train, validation = train_test_split(
        train_validation,
        test_size=adjusted_validation_size,
        random_state=RANDOM_STATE,
        stratify=train_validation[TARGET],
    )

pd.DataFrame(
    {
        "split": ["train", "validation", "test"],
        "rows": [len(train), len(validation), len(test)],
        "default_rate": [train[TARGET].mean(), validation[TARGET].mean(), test[TARGET].mean()],
    }
)

,split,rows,default_rate
0,train,215257,0.080727
1,validation,46127,0.080734
2,test,46127,0.080734


## Impute and Encode

Imputation values are learned from the training split only.

In [26]:
protected_columns_list = [ID_COLUMN, TARGET]
feature_columns = [column for column in train.columns if column not in protected_columns_list]

numeric_columns = train[feature_columns].select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_columns = train[feature_columns].select_dtypes(include=["object"]).columns.tolist()

numeric_medians = train[numeric_columns].median(numeric_only=True).to_dict()
categorical_modes = {
    column: (train[column].mode(dropna=True).iloc[0] if not train[column].mode(dropna=True).empty else "Unknown")
    for column in categorical_columns
}


def transform(split: pd.DataFrame) -> pd.DataFrame:
    result = split.copy()
    for column, value in numeric_medians.items():
        result[column] = result[column].fillna(value)
    for column, value in categorical_modes.items():
        result[column] = result[column].fillna(value)
    return result

train_clean = transform(train)
validation_clean = transform(validation)
test_clean = transform(test)

combined = pd.concat(
    [
        train_clean.assign(__split="train"),
        validation_clean.assign(__split="validation"),
        test_clean.assign(__split="test"),
    ],
    ignore_index=True,
)
combined = pd.get_dummies(combined, columns=categorical_columns, dummy_na=False)

train_encoded = combined[combined["__split"] == "train"].drop(columns="__split")
validation_encoded = combined[combined["__split"] == "validation"].drop(columns="__split")
test_encoded = combined[combined["__split"] == "test"].drop(columns="__split")

train_labels = train_encoded[[ID_COLUMN, TARGET]].copy()
validation_labels = validation_encoded[[ID_COLUMN, TARGET]].copy()
test_labels = test_encoded[[ID_COLUMN, TARGET]].copy()

train_features = train_encoded.drop(columns=[ID_COLUMN, TARGET])
validation_features = validation_encoded.drop(columns=[ID_COLUMN, TARGET])
test_features = test_encoded.drop(columns=[ID_COLUMN, TARGET])

train_features.shape, validation_features.shape, test_features.shape

((215257, 237), (46127, 237), (46127, 237))

## Quality Checks

In [27]:
model_feature_columns = train_features.columns.tolist()
checks = pd.DataFrame(
    [
        ("train_missing_values", int(train_features[model_feature_columns].isna().sum().sum())),
        ("validation_missing_values", int(validation_features[model_feature_columns].isna().sum().sum())),
        ("test_missing_values", int(test_features[model_feature_columns].isna().sum().sum())),
        ("final_feature_count", len(model_feature_columns)),
    ],
    columns=["check", "value"],
)
checks

,check,value
0,train_missing_values,0
1,validation_missing_values,0
2,test_missing_values,0
3,final_feature_count,237


### How To Use This Output

Features near the top are the ones the baseline model used most often to split applicants into lower-risk and higher-risk groups.

If external score features such as `EXT_SOURCE_1`, `EXT_SOURCE_2`, `EXT_SOURCE_3`, or `EXT_SOURCE_MEAN` appear near the top, that confirms the EDA insight that external scores are strong risk signals.

If ratio features such as `CREDIT_INCOME_RATIO` or `ANNUITY_INCOME_RATIO` appear near the top, that suggests affordability is important for default prediction.

## Save Phase 2 Outputs

In [28]:
train_features.to_csv(OUTPUT_DIR / "train_features.csv", index=False)
validation_features.to_csv(OUTPUT_DIR / "validation_features.csv", index=False)
test_features.to_csv(OUTPUT_DIR / "test_features.csv", index=False)

train_labels.to_csv(OUTPUT_DIR / "train_labels.csv", index=False)
validation_labels.to_csv(OUTPUT_DIR / "validation_labels.csv", index=False)
test_labels.to_csv(OUTPUT_DIR / "test_labels.csv", index=False)

pd.DataFrame({"feature": model_feature_columns}).to_csv(OUTPUT_DIR / "feature_columns.csv", index=False)

summary = {
    "dropped_high_missing_columns": dropped_columns,
    "encoded_categorical_columns": categorical_columns,
    "numeric_columns_before_encoding": numeric_columns,
    "final_feature_count": len(model_feature_columns),
    "train_rows": int(len(train_features)),
    "validation_rows": int(len(validation_features)),
    "test_rows": int(len(test_features)),
    "train_default_rate": float(train_labels[TARGET].mean()),
    "validation_default_rate": float(validation_labels[TARGET].mean()),
    "test_default_rate": float(test_labels[TARGET].mean()),
}
(OUTPUT_DIR / "preparation_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")

print(f"Phase 2 feature and label outputs written to: {OUTPUT_DIR}")

Phase 2 feature and label outputs written to: outputs\phase_2
